Модель с архитектурой на основе трансформера будет принимать на вход склеенные через специальный токен название и описание фильма и предсказывать вероятность каждого жанра. 

In [1]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import pytorch_lightning as pl

from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, DataCollatorWithPadding
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from torchmetrics.classification import MultilabelF1Score
from transformers import AutoModel
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger


Для начала напишу датасет, который просто возвращает данные из нужной строчки моего data.csv, при этом так как хочется чтобы модель умела предсказывать жанры без описания фильма (только по названию), добавим dropout: пусть с некоторой вероятностью описание заменяется на пустую строку.

In [2]:

class MoviesDataset(Dataset):
    def __init__(self, df, genre_columns, overview_dropout=0.0, is_train=False):
        self.df = df.reset_index(drop=True)
        self.genre_columns = genre_columns
        self.overview_dropout = overview_dropout
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        title = str(row["title"])
        overview = str(row["overview"])

        if self.is_train and self.overview_dropout > 0:
            if np.random.rand() < self.overview_dropout:
                overview = ""

        labels = row[self.genre_columns].to_numpy(dtype=np.float32)

        return {"title": title, "overview": overview, "labels": labels}

Я буду делать через лайтнинг, потому что так научили в вузе и мне это нравится.

Добавим класс который умеет токенизировать класс, делая паддинг по самому длинному в батче

In [3]:
class MoviesCollator:
    def __init__(self, tokenizer, max_length: int):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, features):
        titles = [f["title"] for f in features]
        overviews = [f["overview"] for f in features]

        batch = self.tokenizer(
            titles,
            overviews,
            truncation=True,
            max_length=self.max_length,
            padding="longest",
            return_tensors="pt",
        )
        labels_np = np.stack([f["labels"] for f in features]).astype(np.float32)
        batch["labels"] = torch.from_numpy(labels_np)
        return batch

Добавляю dataloader который использует класс выше

In [4]:
class MoviesDataLoader(DataLoader):
    def __init__(
        self,
        dataset,
        tokenizer,
        max_length: int,
        batch_size: int,
        shuffle: bool,
        num_workers: int,
    ):
        self.collator = MoviesCollator(tokenizer, max_length=max_length)
        super().__init__(
            dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
            collate_fn=self.collator,
        )

Датамодуль, который использует deberta-v3 и ее токенизатор

In [ ]:

class MoviesDataModule(pl.LightningDataModule):
    def __init__(
        self,
        data_path="data.csv",
        model_name="microsoft/deberta-v3-base",
        batch_size=16,
        max_length=256,
        val_size=0.2,
        seed=42,
        num_workers=4,
        overview_dropout=0.3,
    ):
        super().__init__()
        self.data_path = data_path
        self.model_name = model_name
        self.batch_size = batch_size
        self.max_length = max_length
        self.val_size = val_size
        self.seed = seed
        self.num_workers = num_workers
        self.overview_dropout = overview_dropout

        self.tokenizer = None
        self.genre_columns = None
        self.train_ds = None
        self.val_ds = None

    def setup(self, stage=None):
        df = pd.read_csv(self.data_path)

        self.genre_columns = df.columns[2:].tolist()
        Y = df[self.genre_columns].to_numpy(dtype=np.int64)

        splitter = MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=self.val_size,
            random_state=self.seed,
        )
        train_idx, val_idx = next(splitter.split(df, Y))

        df_train = df.iloc[train_idx].reset_index(drop=True)
        df_val   = df.iloc[val_idx].reset_index(drop=True)

        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name, use_fast=False)

        self.train_ds = MoviesDataset(
            df_train, self.genre_columns,
            overview_dropout=self.overview_dropout,
            is_train=True
        )
        self.val_ds = MoviesDataset(
            df_val, self.genre_columns,
            overview_dropout=0.0,
            is_train=False
        )

    def train_dataloader(self):
        return MoviesDataLoader(
            self.train_ds,
            tokenizer=self.tokenizer,
            max_length=self.max_length,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
        )

    def val_dataloader(self):
        return MoviesDataLoader(
            self.val_ds,
            tokenizer=self.tokenizer,
            max_length=self.max_length,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
        )

Использую deberta-v3-base, не буду ее замораживать полностью, просто обучать с меньшим lr чем голову классификации.

In [6]:
class DebertaGenresModule(pl.LightningModule):
    def __init__(
        self,
        model_name: str = "microsoft/deberta-v3-base",
        num_labels: int = 20,
        lr_backbone: float = 1e-5,
        lr_head: float = 5e-5,
        weight_decay: float = 0.01,
        threshold: float = 0.5,
        head_dropout: float = 0.1,
    ):
        super().__init__()
        self.lr_backbone = lr_backbone
        self.lr_head = lr_head
        self.weight_decay = weight_decay

        self.backbone = AutoModel.from_pretrained(model_name, torch_dtype=torch.float32)
        self.backbone.train()

        hidden = self.backbone.config.hidden_size

        self.dropout = nn.Dropout(head_dropout)
        self.classifier = nn.Linear(hidden, num_labels)

        self.loss_fn = nn.BCEWithLogitsLoss()

        self.train_f1 = MultilabelF1Score(num_labels=num_labels, average="micro", threshold=threshold)
        self.val_f1 = MultilabelF1Score(num_labels=num_labels, average="micro", threshold=threshold)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls_vec = out.last_hidden_state[:, 0, :]
        cls_vec = self.dropout(cls_vec)
        logits = self.classifier(cls_vec)
        return logits

    def training_step(self, batch, batch_idx):
        logits = self(batch["input_ids"], batch["attention_mask"])
        labels = batch["labels"]
        loss = self.loss_fn(logits, labels)

        probs = torch.sigmoid(logits)
        self.train_f1.update(probs, labels.int())

        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log("train_micro_f1", self.train_f1, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        logits = self(batch["input_ids"], batch["attention_mask"])
        labels = batch["labels"]
        loss = self.loss_fn(logits, labels)

        probs = torch.sigmoid(logits)
        self.val_f1.update(probs, labels.int())

        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val_micro_f1", self.val_f1, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            [
                {"params": self.backbone.parameters(), "lr": self.lr_backbone},
                {"params": self.classifier.parameters(), "lr": self.lr_head},
            ],
            weight_decay=self.weight_decay,
        )
        return optimizer

Теперь просто обучаю все это и сохраняю лучшую по метрике f1 модель

In [ ]:
logger = TensorBoardLogger(save_dir="logs", name="tmdb_deberta")

ckpt = ModelCheckpoint(
    dirpath="checkpoints",
    filename="deberta-{epoch:02d}-{val_micro_f1:.4f}",
    monitor="val_micro_f1",
    mode="max",
    save_top_k=1,
    save_last=True,
)

early_stop = EarlyStopping(
    monitor="val_micro_f1",
    mode="max",
    patience=2,
)

lr_monitor = LearningRateMonitor(logging_interval="epoch")

trainer = pl.Trainer(
    max_epochs=30,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    precision="16-mixed" if torch.cuda.is_available() else "32",
    logger=logger,
    callbacks=[ckpt, early_stop, lr_monitor],
    gradient_clip_val=1.0,
    accumulate_grad_batches=1,
    log_every_n_steps=10,
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [8]:
print("cuda available:", torch.cuda.is_available())


cuda available: False


In [ ]:
dm = MoviesDataModule(batch_size=16, max_length=256, num_workers=4)
model = DebertaGenresModule(num_labels=20)

trainer.fit(model, datamodule=dm)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ backbone   │ DebertaV2Model    │  183 M │ train │     0 │
│ 1 │ dropout    │ Dropout           │      0 │ train │     0 │
│ 2 │ classifier │ Linear            │ 15.4 K │ train │     0 │
│ 3 │ loss_fn    │ BCEWithLogitsLoss │      0 │ train │     0 │
│ 4 │ train_f1   │ MultilabelF1Score │      0 │ train │     0 │
│ 5 │ val_f1     │ MultilabelF1Score │      0 │ train │     0 │
└───┴────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 183 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 183 M                                                                                                
Total estimated model params size (MB): 735                                                                        
Modules in train mode: 242                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/slonik11111111/shad/ML2/venv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.


Detected KeyboardInterrupt, attempting graceful shutdown ...
